# Lecture 1

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv('https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/weatherHistory.csv')
df

## ChatGPT output with no guidance

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Parse date column into useful features
df["Formatted Date"] = pd.to_datetime(df["Formatted Date"], utc=True)

df["year"] = df["Formatted Date"].dt.year
df["month"] = df["Formatted Date"].dt.month
df["day"] = df["Formatted Date"].dt.day
df["hour"] = df["Formatted Date"].dt.hour

def run_RF(df):
    # Target column
    y = df["Summary"]
    
    # Drop target and columns that may leak text information
    X = df.drop(columns=["Summary", "Formatted Date", "Daily Summary"])
    
    # Identify feature types
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
    categorical_features = X.select_dtypes(include=["object"]).columns
    
    # Set Up Preprocessing
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
    
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )
    
    # Set Up Random Forest model
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced_subsample",
            n_jobs=-1
        ))
    ])
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Evaluate
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # Save trained model
    #joblib.dump(model, "weather_summary_random_forest.pkl")
    #print("\nModel saved as weather_summary_random_forest.pkl")

run_RF(df)

Let's help ChatGPT out a bit. The error comes from some categories that have too few data points. Before we got started, we should have checked what data catigories we had.

In [ ]:
counts = df["Summary"].value_counts()
counts

There are three catigories with just one entry. Let's remove those.

In [ ]:
df = df[df["Summary"].isin(counts[counts >= 2].index)]
run_RF(df)

In [ ]:
df.dtypes

It worked! However, ChatGPT would not hold up to a peer review. If I was a reviewer on this model, I would ask the following questions.

1. Are all of these categories necessary and descriptive of our weather system?
2. Are the features the best variables needed to describe the weather?
3. All the features were not included because numeric catigories don't cover all data times.
4. Why were day, month, and year coded into the model directly?
5. Why were hyperparameters not tuned?
6. Why was there no iterative feature selection?
7. Cloud cover was included as a feature, but there was no variance in the data. 